## Data Manupilation

In [ ]:
import polars as pl

In [ ]:
column_rename_mapping = {
    "VendorID": "vendor_id",
    "RatecodeID": "rate_code_id",
    "PULocationID": "pickup_location_id",
    "DOLocationID": "dropoff_location_id",
    "payment_type": "payment_type",
}
df = pl.read_parquet("./../data/yellow_tripdata_2024-03.parquet").rename(
    column_rename_mapping
)

In [ ]:
df.head()

In [ ]:
df.group_by("pickup_location_id").agg(
    pl.len().alias("rows"),
    pl.col("passenger_count").max().name.suffix("_max"),
    pl.col("trip_distance").min().name.suffix("_min"),
    pl.col("trip_distance").mean().name.suffix("_mean"),
    pl.col("trip_distance").max().name.suffix("_max"),
).sort(pl.col("rows"), descending=True)

In [ ]:
df.group_by(
    pl.col("pickup_location_id")
    .eq(pl.col("dropoff_location_id"))
    .alias("same_location")
).agg(
    pl.len().alias("count_trips"),
    pl.col("passenger_count").max().name.suffix("_max"),
    pl.col("trip_distance").min().name.suffix("_min"),
    pl.col("trip_distance").mean().name.suffix("_mean"),
    pl.col("trip_distance").max().name.suffix("_max"),
).sort("count_trips", descending=True)

## Window Functions / .over method

In [ ]:
df.with_columns(pl.col("tip_amount").gt(0).alias("had_tip")).sort(
    pl.col("trip_distance"), descending=True
).select(
    "trip_distance",
    "tip_amount",
    "had_tip",
)

In [ ]:
df.with_columns(pl.col("tip_amount").gt(0).alias("had_tip")).sort(
    pl.col("trip_distance"), descending=True
).select(
    "trip_distance",
    "tip_amount",
    "had_tip",
    pl.col("trip_distance").rank(descending=True).name.suffix("_rank"),
)

In [ ]:
df.with_columns(pl.col("tip_amount").gt(0).alias("had_tip")).sort(
    pl.col("trip_distance"), descending=True
).select(
    "trip_distance",
    "tip_amount",
    "had_tip",
    pl.col("trip_distance")
    .rank(descending=True)
    .over("had_tip")
    .name.suffix("_rank_with_had_tip"),
)

In [ ]:
df.sort(pl.col("trip_distance"), descending=True).select(
    "trip_distance",
    "tip_amount",
    pl.col("trip_distance")
    .rank(descending=True)
    .over(pl.col("tip_amount").gt(0))
    .name.suffix("_rank_within_had_tip"),
).filter(pl.col("trip_distance_rank_within_had_tip").le(3))

## Pivot

In [ ]:
df.with_columns(pl.col("tolls_amount").gt(0).alias("had_toll")).pivot(
    index="had_toll",
    on="passenger_count",
    values="tip_amount",
    aggregate_function="mean",
    sort_columns=True,
)

In [ ]:
df.with_columns(pl.col("tolls_amount").gt(0).alias("had_toll")).pivot(
    index="had_toll",
    on="passenger_count",
    values="tip_amount",
    aggregate_function="len",
    sort_columns=True,
)

In [ ]:
df.with_columns(pl.col("tolls_amount").gt(0).alias("had_toll")).pivot(
    index=["pickup_location_id", "dropoff_location_id"],
    on="passenger_count",
    values="tip_amount",
    aggregate_function="len",
    sort_columns=True,
)

In [ ]:
df